# Infection-Triggered Encephalopathy Syndromes: Re-classification of the Acute Encephalopathy Cohort

**Purpose.** Re-classify every index admission of the acute encephalopathy cohort against the 2024 international
consensus definitions of infection-triggered encephalopathy syndromes (ITES), produce an adjudication worksheet for
pediatric neurologists, and summarize the reclassified cohort.

**How to run.** Place this notebook in the same folder as the two research-platform exports (the main export with sheet
`病案首页基本信息` and the supplementary export with sheet `病案首页其他信息`) and the frozen dataset folder `AE_freeze_v1.2`
(reference labels for discharge disposition and organ support), and run all cells. Outputs are written to
`./output_ites/`; the parsed exports are cached in `./cache_aes_exports/` (shared with the other notebooks).

**Adjudication loop.** Fill in column `adjudicated_class` of `output_ites/ITES_adjudication_worksheet.xlsx`, save the
columns `vis` and `adjudicated_class` as `output_ites/ites_adjudicated.csv` (UTF-8), and re-run all notebooks:
adjudicated categories then replace the algorithmic ones everywhere.

**Data notes.** Classification uses only exported structured fields and report text (discharge diagnoses, MRI/CT/EEG
reports, CSF and blood tests, admission notes, vital signs, medication orders). Free text is used for rule matching only;
exported files contain research identifiers, flags, and short report phrases, never names.

In [ ]:
import re, json, warnings, collections
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
SEED = 20261009
rng = np.random.default_rng(SEED)
HSCTN_URL = 0.014       # ng/mL: 99th-percentile URL of the hs-cTnT assay (Roche cobas e602; report displays 0.01 after rounding)
HSCTN_URL_ALT = 2 * HSCTN_URL   # ng/mL: sensitivity threshold (twice the URL)
URL_NGL, ALT_NGL = int(round(HSCTN_URL * 1000)), int(round(HSCTN_URL_ALT * 1000))
from decimal import Decimal, ROUND_HALF_UP

def rnd(x, d=1):
    # Conventional half-up rounding for displayed numbers (avoids binary-float and half-to-even artifacts).
    return str(Decimal(f'{float(x):.10f}').quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP))

HERE = Path.cwd()
OUT = HERE / 'output_ites'
CACHE = HERE / 'cache_aes_exports'
FIG = OUT / 'figures'
for p in (OUT, CACHE, FIG):
    p.mkdir(exist_ok=True)

def pick_font():
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']:
        if name in available:
            return name
    return 'sans-serif'

FONT = pick_font()
plt.rcParams.update({
    'font.family': FONT, 'font.size': 13, 'axes.labelsize': 14, 'axes.labelweight': 'bold',
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 11, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False, 'axes.linewidth': 1.0,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
print('Figure font:', FONT, '(Arial is used whenever it is installed)')

In [ ]:
# Locate and load both exports (about 2.5 minutes on the first run; cached afterwards).
KEY_SHEET, SUPP_SHEET = '病案首页基本信息', '病案首页其他信息'

def find_workbooks(folder):
    main = supp = None
    for p in sorted(folder.glob('*.xlsx')):
        if p.name.startswith('~$'):
            continue
        try:
            wb = openpyxl.load_workbook(p, read_only=True); names = wb.sheetnames; wb.close()
        except Exception:
            continue
        if SUPP_SHEET in names:
            supp = p
        elif KEY_SHEET in names:
            main = p
    if main is None or supp is None:
        raise FileNotFoundError('Both the main export and the supplementary export must be in this folder.')
    return main, supp

def load_book(path, cache_name):
    f = CACHE / cache_name
    if f.exists() and f.stat().st_mtime > path.stat().st_mtime:
        return pd.read_pickle(f)
    book = pd.read_excel(path, sheet_name=None, dtype=object, engine='openpyxl')
    pd.to_pickle(book, f)
    return book

WB, WB_SUPP = find_workbooks(HERE)
sheets = load_book(WB, 'main.pkl')
supp = load_book(WB_SUPP, 'supplement.pkl')
print('Main export:', WB.name, '| sheets:', len(sheets))
print('Supplementary export:', WB_SUPP.name, '| sheets:', len(supp))

In [ ]:
# Admissions and free-text discharge diagnoses (same parser as the data-asset notebook).
def split_top(s):
    out, buf, depth = [], '', 0
    for ch in s:
        if ch in '(（[［':
            depth += 1
        elif ch in ')）]］':
            depth = max(0, depth - 1)
        if ch in ',，;；、' and depth == 0:
            out.append(buf); buf = ''
        else:
            buf += ch
    out.append(buf)
    return out

def parse_dx(s, kind='出院'):
    if not isinstance(s, str):
        return []
    s = re.sub(kind + r'诊断\s*\d*\s*[:：]', '\n', s.replace('\r', '\n'))
    items = []
    for line in s.split('\n'):
        line = line.strip()
        if not line:
            continue
        for part in re.split(r'(?:(?<=^)|(?<=[^\d\.]))\d{1,2}\s*[\.．、]\s*(?=\D)', line):
            for q in split_top(part):
                q = q.strip(' 　\t。.')
                if q:
                    items.append(q)
    return items

def uniq(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

fp = sheets['病案首页基本信息'].copy()
fp['vis'] = fp['科研就诊编号'].astype(str)
fp['pid'] = fp['科研患者编号'].astype(str)
fp['admit'] = pd.to_datetime(fp['入院日期'], errors='coerce')
fp['discharge'] = pd.to_datetime(fp['出院日期'], errors='coerce')
fp['age_y'] = pd.to_numeric(fp['年龄（岁）'], errors='coerce')

dis = sheets['住院病历出院记录'].copy()
dis['dx_list'] = dis['出院诊断'].apply(parse_dx)
longest = lambda s: max([x for x in s if isinstance(x, str)] or [''], key=len)
agg = dis.groupby('科研就诊编号').agg(dx_list=('dx_list', lambda s: uniq(sum(s.tolist(), []))),
                                    course=('诊疗经过', longest), dstatus=('出院情况', longest)).reset_index()
agg['vis'] = agg['科研就诊编号'].astype(str)
adm = fp.merge(agg.drop(columns=['科研就诊编号']), on='vis', how='left')
adm['has_disrec'] = adm['dx_list'].apply(lambda x: isinstance(x, list))
adm['dx_list'] = adm['dx_list'].apply(lambda x: x if isinstance(x, list) else [])
for c in ['course', 'dstatus']:
    adm[c] = adm[c].fillna('')
print('Admissions:', len(adm), '| patients:', adm.pid.nunique())

In [ ]:
# Provisional syndrome labels (rule-based; identical to the data-asset notebook).
ORDER = [
    ('Metabolic_Leigh', r'亚急性坏死性|Leigh'),
    ('ANE', r'坏死性\)?脑|脑病[^,，;；]{0,14}坏死性|急性坏死性(?!.{0,4}(?:肠|筋膜|牙龈|肺|淋巴|胰))'),
    ('HSES_ASEM', r'出血性休克脑病|休克脑病|HSES'),
    ('Reye', r'瑞氏|Reye|瑞夷'),
    ('AESD', r'AESD|双相发作|双向发作|迟发性弥散|后期弥散|后期弥漫|晚期弥散|迟发性扩散'),
    ('FIRES', r'FIRES|FIERS|热性感染相关|发热感染相关|热性敏感性|感染相关性癫痫综合征|AERRPS|难治性反复部分性'),
    ('MERS', r'胼胝体压部|MERS|RESLES|胼胝体压迫|可逆性胼胝体|胼胝体可逆'),
    ('Septic_encephalopathy', r'脓毒症性\)?脑病|脓毒性\)?脑病|脓毒症相关性脑病|脓毒症脑病'),
    ('Pertussis_encephalopathy', r'百日咳脑病'),
    ('Influenza_encephalopathy', r'流感相关性\)?脑病|流感脑病|流感相关脑病|流感相关性脑炎'),
    ('Toxic_encephalopathy', r'中毒性脑病'),
    ('Bilirubin', r'胆红素脑病|核黄疸'),
    ('Neonatal_HIE_NE', r'新生儿.*脑病|新生儿缺氧缺血|新生儿缺血缺氧|\(新生儿\)缺氧'),
    ('HIE_other', r'缺氧缺血性脑病|缺血缺氧性脑病|缺氧性脑病|缺氧缺血性脑损伤'),
    ('DEE', r'癫痫性脑病|ESES|CSWS|睡眠中癫痫性电持续|Lennox|大田原|暴发抑制|发育性.*脑病|癫痫性\)脑病|West综合征|婴儿痉挛'),
    ('Metabolic', r'代谢性脑病|甘氨酸脑病|低血糖性脑病|低血糖昏迷|肝性脑病|尿毒症性脑病|Wernicke|韦尼克|线粒体脑|高氨血症脑病'),
    ('PRES_hypertensive', r'可逆性后部|PRES|高血压脑病|后部可逆'),
    ('Lupus', r'狼疮性脑病|狼疮脑病|神经精神狼疮|狼疮脑'),
    ('Autoimmune', r'自身免疫性脑炎|自体免疫性脑炎|NMDA|GABA|LGI1|免疫介导性脑炎|自身免疫相关性脑病|桥本脑病|自身免疫性\)?脑病|免疫性？\)|ADEM|急性播散性脑脊髓炎'),
    ('Infectious_encephalitis', r'病毒性脑炎|脑膜脑炎|^\(?病毒性\)?脑炎|化脓性脑膜炎|细菌性脑膜炎|结核性脑膜炎|^脑炎|汉坦病毒脑炎|疱疹病毒性脑炎|乙型脑炎'),
    ('Leukoencephalopathy', r'白质脑病|脑白质病'),
    ('AES_unspecified', r'急性脑病|急性\)\s*脑病|^脑病|弥散性脑病|中枢神经系统感染相关性脑病'),
]
AES = ['ANE', 'HSES_ASEM', 'Reye', 'AESD', 'FIRES', 'MERS', 'Septic_encephalopathy', 'Pertussis_encephalopathy',
       'Influenza_encephalopathy', 'Toxic_encephalopathy', 'AES_unspecified']
SEQ = re.compile(r'后遗症|恢复期|随诊|复查|治疗后|个人史')

def label_string(x):
    if '胼胝体' in x and re.search(r'先天性|畸形|发育不良|缺如', x) and '脑病' not in x:
        return None
    for k, pat in ORDER:
        if re.search(pat, x):
            return k
    return None

def visit_labels(L):
    out = {}
    for x in L:
        k = label_string(x)
        if k:
            out.setdefault(k, []).append(x)
    return out

def aes_info(lab):
    keys = [k for k in lab if k in AES]
    if not keys:
        return pd.Series([False, None, None])
    strs = sum([lab[k] for k in keys], [])
    acute = any(not SEQ.search(s) for s in strs)
    return pd.Series([True, next(k for k in AES if k in keys), acute])

# Coded discharge diagnoses are used only where no discharge summary exists (mostly in-hospital deaths).
dxc = supp['病案出院诊断归一'].copy(); dxc['vis'] = dxc['科研就诊编号'].astype(str)
coded = dxc.groupby('vis')['诊断疾病名称'].apply(lambda s: [str(x) for x in s])
adm['coded_list'] = adm.vis.map(coded).apply(lambda x: x if isinstance(x, list) else [])
adm['lab'] = [visit_labels(L if hd else C) for L, C, hd in zip(adm['dx_list'], adm['coded_list'], adm['has_disrec'])]
adm[['aes', 'aes_sub', 'aes_acute']] = adm['lab'].apply(aes_info)
adm['all_dx_text'] = [' | '.join(L + C) for L, C in zip(adm['dx_list'], adm['coded_list'])]
print('AES-labeled admissions:', int(adm.aes.sum()))

In [ ]:
# Index admission: the first acute-phase AES admission per patient; dispositions from the supplement.
oth = supp['病案首页其他信息'].copy(); oth['vis'] = oth['科研就诊编号'].astype(str)
PICU_WARD = re.compile(r'重症医学|儿童重症监护')     # PICU wards (surgical ICU not included)
trf = supp['患者转科信息'].copy(); trf['vis'] = trf['科研就诊编号'].astype(str)
trf['t'] = pd.to_datetime(trf['转科时间'], errors='coerce')
trf = trf.merge(adm[['vis', 'admit', 'discharge']], on='vis', how='inner')
trf = trf[(trf.t >= trf.admit - pd.Timedelta(days=1)) & (trf.t <= trf.discharge + pd.Timedelta(days=1))]
PICU_TRANSFER = set(trf[trf['转出科室'].astype(str).str.contains(PICU_WARD) | trf['转入科室'].astype(str).str.contains(PICU_WARD)].vis)
vinfo = supp['患者就诊信息'].copy(); vinfo['vis'] = vinfo['科研就诊编号'].astype(str)
adm = adm.merge(oth[['vis', '离院方式']], on='vis', how='left').merge(vinfo[['vis', '转归情况']], on='vis', how='left')
adm['covered'] = adm.vis.isin(set(oth.vis))
adm['death'] = adm['离院方式'].eq('死亡') | adm['转归情况'].eq('死亡')
adm['ama'] = adm['离院方式'].eq('非医嘱离院')

# Critical-condition wording at the end of the discharge summary (supplement version, else main export).
WD = re.compile(r'放弃治疗|放弃抢救|自动出院|拒绝.{0,6}(?:气管插管|治疗)|随时.{0,6}死亡|死亡风险|心跳骤停.{0,10}(?:风险|可能)|极可能.{0,4}死亡|带气管插管出院|拔除气管插管|撤机')
CRIT = re.compile(r'危重|病重|预后差|预后极差|预后不良|昏迷|电静息|脑死亡|脑疝')
ds = supp['住院病历出院记录'].copy(); ds['vis'] = ds['科研就诊编号'].astype(str)
ds_txt = ds.groupby('vis').apply(lambda g: ' '.join(str(x) for x in list(g['诊疗经过'])[-1:] + list(g['出院情况'])))
def critical_wording(row):
    t = ds_txt.get(row.vis, None)
    t = str(t) if t is not None else (row.course + ' ' + row.dstatus)
    tail = t[-600:]
    return bool(WD.search(tail) and CRIT.search(tail))

adm = adm.sort_values(['pid', 'admit'])
rows = []
for pid, g in adm[adm.aes == True].groupby('pid'):
    sub = next(k for k in AES if k in set(g.aes_sub))
    ac = g[g.aes_acute == True]
    r = ac.iloc[0] if len(ac) else g.iloc[0]
    wards = ' '.join(str(x) for x in [r['入院科别'], r['出院科别'], r['转科科别']] if isinstance(x, str))
    rows.append({'patient': pid, 'vis': r.vis, 'subtype': sub, 'acute_index': bool(r.aes_acute), 'admit': r.admit,
                 'discharge': r.discharge, 'year': r.admit.year, 'age_y': r.age_y, 'male': r['性别'] == '男性',
                 'icu': bool(PICU_WARD.search(wards)) or r.vis in PICU_TRANSFER, 'covered': bool(r.covered), 'death': bool(r.death), 'ama': bool(r.ama),
                 'disposition': r['离院方式'], 'has_disrec': bool(r.has_disrec), 'dx_text': r.all_dx_text,
                 'course': r.course, 'dstatus': r.dstatus})
coh = pd.DataFrame(rows)
idx = coh[coh.acute_index & coh.covered].copy().reset_index(drop=True)
idx['critical_ama_rule'] = idx.ama & idx.apply(critical_wording, axis=1)   # documentation rule (sensitivity label)

# Reference labels from the frozen dataset of this export (AE_freeze_v1.2): critical discharge against medical advice as adjudicated by a
# pediatric clinician (dama_critical_final; the freeze rule label is kept where the reviewer was unsure).
FREEZE_DIR = HERE / 'AE_freeze_v1.2'
if not (FREEZE_DIR / 'tables' / 'admission_summary.csv').exists():
    raise FileNotFoundError('Place the frozen dataset folder AE_freeze_v1.2 (with its tables/ subfolder) next to this notebook.')
FZ = pd.read_csv(FREEZE_DIR / 'tables' / 'admission_summary.csv', encoding='utf-8-sig', dtype={'visit_id': str}, low_memory=False).set_index('visit_id')
FZ_EVENTS = pd.read_csv(FREEZE_DIR / 'tables' / 'support_events.csv', encoding='utf-8-sig', dtype={'visit_id': str}, low_memory=False)
not_in_freeze = set(idx.vis) - set(FZ.index)
if not_in_freeze:
    raise ValueError(f'{len(not_in_freeze)} index admissions are missing from the frozen dataset.')

def fz_flag(col):
    return idx.vis.map(FZ[col].astype(str).str.lower().eq('true')).astype(bool)

idx['critical_ama'] = idx.ama & fz_flag('dama_critical_final')
idx['poor'] = idx.death | idx.critical_ama
print('AES patients:', len(coh), '| acute index admissions:', int(coh.acute_index.sum()),
      '| with a recorded disposition:', len(idx))
print('Deaths:', int(idx.death.sum()), '| critical AMA (clinician-adjudicated):', int(idx.critical_ama.sum()),
      '| by the documentation rule:', int(idx.critical_ama_rule.sum()), '| death or critical AMA:', int(idx.poor.sum()))

In [ ]:
# Laboratory, ECG, and echocardiography measurements (both exports, de-duplicated by report number).
def parse_lab_value(quant, qual=None):
    # Numeric result; when only the qualitative column holds it (for example "<0.003" or ">9000"), the value or the
    # reportable limit is used and the result is flagged as censored.
    v = pd.to_numeric(quant, errors='coerce')
    if qual is None:
        return v, pd.Series(False, index=v.index)
    q = qual.astype(str).str.strip().str.replace('〈', '<').str.replace('〉', '>').str.replace('＜', '<').str.replace('＞', '>')
    m = q.str.extract(r'^([<>]=?)?\s*(\d*\.?\d+)$')
    qv = pd.to_numeric(m[1], errors='coerce')
    use = v.isna() & qv.notna()
    return v.where(~use, qv), use & m[0].notna()

def qual_col(df, col):
    base = col.replace('-定量结果', '')
    for c in (base + '-定性结果', base + '-定性结果显示'):
        if c in df.columns:
            return df[c]
    return None

def lab_long(sheet, col):
    parts = []
    for book in (sheets, supp):
        if sheet in book and col in book[sheet].columns:
            df = book[sheet]
            val, cens = parse_lab_value(df[col], qual_col(df, col))
            parts.append(pd.DataFrame({'report': df['报告编号'].astype(str), 'vis': df['就诊编号'].astype(str),
                                       'value': val, 'censored': cens,
                                       'time': pd.to_datetime(df['报告时间'], errors='coerce')}))
    x = pd.concat(parts).dropna(subset=['value', 'time']).drop_duplicates('report')
    return x[x.vis.isin(set(idx.vis))]

def summarize(long, hours, prefix):
    x = long.merge(idx[['vis', 'admit', 'discharge']], on='vis')
    x['h'] = (x.time - x.admit).dt.total_seconds() / 3600
    within = x[(x.h >= -12) & (x.h <= hours)].sort_values('time')
    first = within.groupby('vis').first()
    out = pd.DataFrame({f'{prefix}_early': first.value, f'{prefix}_early_time': first.time})
    out[f'{prefix}_peak'] = x.groupby('vis').value.max()
    out[f'{prefix}_first_time'] = x.sort_values('time').groupby('vis').time.first()
    return out

tnt = summarize(lab_long('实验室检查_血液心肌酶谱测定', '超敏心肌钙蛋白定量-定量结果'), 48, 'tnt')
ckmb = summarize(lab_long('实验室检查_血液心肌酶谱测定', '肌酸激酶同工酶-ＭＢ定量-定量结果'), 48, 'ckmb')
bnp = summarize(lab_long('实验室检查_血液N末端B型钠尿肽前体测定', 'N末端B型利钠肽原定量-定量结果'), 48, 'bnp')
lac = summarize(lab_long('实验室检查_血液乳酸测定', '乳酸定量-定量结果'), 48, 'lac')

ecg = supp['心电图报告'].copy(); ecg['vis'] = ecg['科研就诊编号'].astype(str)
ecg['time'] = pd.to_datetime(ecg['心电检查时间'], errors='coerce')
ecg['HR'] = pd.to_numeric(ecg['HR'], errors='coerce'); ecg['QT'] = pd.to_numeric(ecg['QT'], errors='coerce')
ecg = ecg[ecg.vis.isin(set(idx.vis)) & ecg.HR.between(30, 300) & ecg.QT.between(150, 700)].copy()
ecg['value'] = ecg.QT / np.cbrt(60 / ecg.HR)                      # Fridericia
ecg['QTcB'] = ecg.QT / np.sqrt(60 / ecg.HR)                        # Bazett (equals the machine QTc)
ecg['report'] = ecg.index.astype(str)
qtc = summarize(ecg[['report', 'vis', 'value', 'time']], 48, 'qtcf')
qtc['qtcb_peak'] = ecg.groupby('vis').QTcB.max()
qtc['qtcb_early'] = summarize(ecg[['report', 'vis', 'QTcB', 'time']].rename(columns={'QTcB': 'value'}), 48, 'qtcb')['qtcb_early']

echo = supp['儿童心脏彩超'].copy(); echo['vis'] = echo['科研就诊编号'].astype(str)
echo['time'] = pd.to_datetime(echo['检查时间'], errors='coerce'); echo['value'] = pd.to_numeric(echo['射血分数'], errors='coerce')
echo = echo[echo.vis.isin(set(idx.vis))].dropna(subset=['value', 'time'])
echo['report'] = echo.index.astype(str)
x = echo.merge(idx[['vis', 'admit']], on='vis'); x['h'] = (x.time - x.admit).dt.total_seconds() / 3600
ef = pd.DataFrame({'lvef_min': x.groupby('vis').value.min(),
                   'lvef_early_min': x[(x.h >= -24) & (x.h <= 72)].groupby('vis').value.min(),
                   'lvef_first_time': x.sort_values('time').groupby('vis').time.first()})

for t in (tnt, ckmb, bnp, lac, qtc, ef):
    idx = idx.merge(t, left_on='vis', right_index=True, how='left')
idx = idx.reset_index(drop=True)
idx['tnt_early_ngL'] = idx.tnt_early * 1000
idx['tnt_peak_ngL'] = idx.tnt_peak * 1000
print('Index admissions with early hs-cTn:', int(idx.tnt_early.notna().sum()), '| NT-proBNP:', int(idx.bnp_early.notna().sum()),
      '| ECG QT/HR:', int(idx.qtcf_early.notna().sum()), '| LVEF:', int(idx.lvef_min.notna().sum()))

In [ ]:
# Organ-support and severity variables, cardiac arrest timing, era.
proc = supp['病案手术操作归一'].copy(); proc['vis'] = proc['科研就诊编号'].astype(str)
proc['date'] = pd.to_datetime(proc['手术操作日期'], errors='coerce')
proc = proc[proc.vis.isin(set(idx.vis))]
code_ = proc['手术操作编码'].astype(str)
vent_proc = set(proc[code_.str.match(r'^96\.7|^96\.04')].vis)
cpr_proc = proc[code_.str.match(r'^99\.6')].groupby('vis').date.min()

orders = supp['住院非用药医嘱'].copy(); orders['vis'] = orders['科研就诊编号'].astype(str)
orders = orders[orders.vis.isin(set(idx.vis))]
vent_order = set(orders[orders['住院非用药医嘱名称'].astype(str).str.contains('气管插管')].vis)

meds = pd.concat([sheets['住院用药医嘱'][['科研就诊编号', '住院用药医嘱药品名称', '住院用药医嘱用法']],
                  supp['住院用药医嘱'][['科研就诊编号', '住院用药医嘱药品名称', '住院用药医嘱用法']]])
meds['vis'] = meds['科研就诊编号'].astype(str)
meds = meds[meds.vis.isin(set(idx.vis))]
name = meds['住院用药医嘱药品名称'].astype(str); route = meds['住院用药医嘱用法'].astype(str)
VASO = r'(?<!异丙)肾上腺素|多巴胺|多巴酚丁胺|米力农|垂体后叶|血管加压素'   # isoproterenol (异丙肾上腺素) excluded
BOLUS_OR_LOCAL = r'雾化|气管|静脉注射|静推|皮下|外用|局部|口服|滴眼|滴鼻'
vaso = set(meds[name.str.contains(VASO) & ~route.str.contains(BOLUS_OR_LOCAL)].vis)
DRUG = {'methylprednisolone': r'甲泼尼龙|甲强龙|甲基强的松龙', 'ivig': r'丙种球蛋白|免疫球蛋白', 'tocilizumab': r'托珠单抗'}
for k, p in DRUG.items():
    idx[k] = idx.vis.isin(set(meds[name.str.contains(p)].vis))

idx['vasoactive'] = idx.vis.isin(vaso)

# Time-resolved organ support (order timestamps) for the time-anchored analyses.
meds_t = pd.concat([sheets['住院用药医嘱'][['科研就诊编号', '住院用药医嘱开立时间', '住院用药医嘱药品名称', '住院用药医嘱用法']],
                    supp['住院用药医嘱'][['科研就诊编号', '住院用药医嘱开立时间', '住院用药医嘱药品名称', '住院用药医嘱用法']]])
meds_t['vis'] = meds_t['科研就诊编号'].astype(str)
meds_t = meds_t[meds_t.vis.isin(set(idx.vis))]
meds_t['t'] = pd.to_datetime(meds_t['住院用药医嘱开立时间'], errors='coerce')
nm, rt = meds_t['住院用药医嘱药品名称'].astype(str), meds_t['住院用药医嘱用法'].astype(str)
infusion = meds_t[nm.str.contains(VASO) & ~rt.str.contains(BOLUS_OR_LOCAL)]
pressor = infusion[~infusion['住院用药医嘱药品名称'].astype(str).str.contains('多巴酚丁胺|米力农')]
idx['vaso_time'] = idx.vis.map(infusion.groupby('vis').t.min())
idx['pressor_time'] = idx.vis.map(pressor.groupby('vis').t.min())
orders['t'] = pd.to_datetime(orders['住院非用药医嘱开始时间'], errors='coerce')
onames = orders['住院非用药医嘱名称'].astype(str)
inv = orders[onames.str.contains('气管插管|呼吸机辅助呼吸|机械通气') & ~onames.str.contains('无创|鼻塞|CPAP|高流量')]
vent_proc_date = proc[code_.str.match(r'^96\.7|^96\.04')].groupby('vis').date.min()
idx['vent_time'] = pd.concat([idx.vis.map(inv.groupby('vis').t.min()), idx.vis.map(vent_proc_date)], axis=1).min(axis=1)
idx['ventilation'] = idx.vis.isin(vent_proc | vent_order | set(inv.vis))
# Reference definitions: vasoactive infusion and invasive ventilation (presence and first time) follow the frozen dataset rules (AE_freeze_v1.2,
# definitions section 3: continuous-infusion routes only, pituitrin reported separately; planned-procedure intubation and ventilator orders with
# non-invasive ventilation evidence not counted). Vasopressor-only start excludes dobutamine and milrinone.
own = idx[['vasoactive', 'ventilation']].copy()
idx['vasoactive'] = fz_flag('vasoactive_ever')
idx['vaso_time'] = pd.to_datetime(idx.vis.map(FZ.vasoactive_first_time), errors='coerce')
va_ev = FZ_EVENTS[FZ_EVENTS.support.eq('vasoactive_infusion')].copy()
va_ev['t'] = pd.to_datetime(va_ev.event_time, errors='coerce')
idx['pressor_time'] = idx.vis.map(va_ev[~va_ev.detail.astype(str).isin(['dobutamine', 'milrinone'])].groupby('visit_id').t.min())
idx['ventilation'] = fz_flag('imv_ever')
idx['vent_time'] = pd.to_datetime(idx.vis.map(FZ.imv_first_time), errors='coerce')
print('Frozen-dataset rules vs order-text rules: vasoactive', int(idx.vasoactive.sum()), 'vs', int(own.vasoactive.sum()),
      '| invasive ventilation', int(idx.ventilation.sum()), 'vs', int(own.ventilation.sum()))
txt = idx.dx_text
idx['status_epilepticus'] = txt.str.contains('癫痫持续状态|惊厥持续状态')
idx['septic_shock'] = txt.str.contains('脓毒性休克|感染性休克')
idx['shock_dx'] = txt.str.contains('休克')
idx['lactate_ge4'] = idx.lac_early >= 4
idx['age_lt1'] = idx.age_y < 1

# Cardiac arrest / CPR: dated procedures, undated diagnoses, or narrative mention.
idx['cpr_date'] = idx.vis.map(cpr_proc)
ARREST = re.compile(r'心肺复苏|心脏骤停|心跳骤停|呼吸心跳骤停|心搏骤停|心脏停搏')
HEDGE = re.compile(r'可能|随时|风险|警惕|预防|避免|防止|告知|交代|谈话|签字|若|如出现|一旦|拒绝|放弃|不予')

def arrest_documented(text):
    # Affirmative narrative mention of cardiac arrest or CPR; clauses with warnings, counseling, or refusal are ignored.
    for sent in re.split(r'[。；;\n！!？?]', str(text)):
        for c in re.split(r'[，,]', sent):
            m = ARREST.search(c)
            if m and not HEDGE.search(c) and not re.search(r'无|未|否认', c[max(0, m.start() - 4):m.start()]):
                return True
    return False

idx['cpr_any'] = idx.cpr_date.notna() | txt.str.contains('心脏停搏|呼吸心跳骤停|心跳骤停|心搏骤停') | idx.course.apply(arrest_documented)
# CPR orders and procedure codes from the frozen dataset are added; the earliest dated evidence is the CPR date.
idx['cpr_any'] = idx.cpr_any | fz_flag('cpr_ever')
idx['cpr_date'] = pd.concat([idx.cpr_date, pd.to_datetime(idx.vis.map(FZ.cpr_first_time), errors='coerce').dt.normalize()], axis=1).min(axis=1)
def arrest_before(first_time):
    # True when an arrest/CPR is documented and cannot be shown to occur on a later calendar day than first_time.
    later = idx.cpr_date.notna() & first_time.notna() & (first_time.dt.normalize() < idx.cpr_date)
    return idx.cpr_any & ~later
idx['arrest_before_tnt'] = arrest_before(idx.tnt_early_time)
idx['arrest_before_echo'] = arrest_before(idx.lvef_first_time)
idx['arrest_before_ecg'] = arrest_before(idx.qtcf_early_time)
idx['arrest_before_bnp'] = arrest_before(idx.bnp_early_time)
idx['arrest_before_tnt_first'] = arrest_before(idx.tnt_first_time)

idx['era'] = pd.cut(idx.admit, [pd.Timestamp('2000-01-01'), pd.Timestamp('2020-01-01'), pd.Timestamp('2022-12-01'),
                                pd.Timestamp('2023-02-01'), pd.Timestamp('2100-01-01')],
                    labels=['Pre-2020', '2020-Nov 2022', 'Dec 2022-Jan 2023', 'Feb 2023 onward'], right=False)
print(idx[['ventilation', 'vasoactive', 'status_epilepticus', 'septic_shock', 'cpr_any', 'methylprednisolone', 'ivig', 'tocilizumab']].sum())

## ITES re-classification (2024 international consensus)

Each index admission is re-classified against the 2024 international consensus definitions of
infection-triggered encephalopathy syndromes (ITES; Sakuma et al., Dev Med Child Neurol 2024).
The algorithm only *proposes* a category from the exported evidence; every proposal is written to an
adjudication worksheet so that a pediatric neurologist can confirm or override it. If an adjudicated file
(`output_ites/ites_adjudicated.csv`, columns `vis` and `adjudicated_class`) is present, adjudicated
categories replace the algorithmic ones in all downstream analyses.

In [ ]:
# ITES evidence 1/4: neuroimaging features from MRI and head CT reports (clause-level negation, side-aware thalamic rule).
SENT = re.compile(r'[。；;\n！!？?]')
CLAUSE = re.compile(r'[，,]')
NEG_IMG = re.compile(r'未见|未发现|无明显|无异常|无确切|未显示|不明显|消失|吸收|未见确切|(?<!异常)正常|排除')
ABN = re.compile(r'异常信号|信号异常|异常密度|长T1|长T2|短T1|短T2|高信号|信号增高|信号灶|信号影|弥散受限|扩散受限|肿胀|肿大|水肿|坏死|出血'
                 r'|异常灶|病灶|低密度|密度减低|密度降低|异常改变')
DWI_POS = re.compile(r'弥散受限|扩散受限|弥散[^，,。；]{0,4}受限|DWI[^，,。；]{0,14}(?:高信号|信号增高|受限)|弥散加权[^，,。；]{0,8}高信号')
SIDE = re.compile(r'双侧|两侧|对称|左|右')
DIFFUSE_EDEMA = re.compile(   # global swelling, herniation or cisternal effacement
    r'全脑(?:肿胀|水肿)|弥漫性脑(?:肿胀|水肿)|广泛(?:性)?脑(?:肿胀|水肿)|脑组织弥漫性?肿胀|颅内广泛脑肿胀|全脑弥漫性肿胀'
    r'|大脑[、及和]?小脑半球[^。；]{0,24}肿胀|扁桃体[^，,。；]{0,6}(?:下疝|疝入)|脑疝形成|全脑.{0,6}脑疝'
    r'|脑池[^，,。；]{0,6}(?:闭塞|消失)|基底池[^，,。；]{0,6}(?:受压|闭塞|消失)'
    r'|脑沟[^，,。；]{0,4}(?:普遍|广泛|弥漫)[^，,。；]{0,4}(?:变浅|消失)|(?:普遍|广泛|弥漫)[^，,。；]{0,6}脑沟[^，,。；]{0,6}(?:变浅|消失)')
GWM_LOSS = re.compile(r'(?:双侧|两侧)大脑(?:半球)?[^。；]{0,16}灰白质(?:分界|界限)[^，,。；]{0,3}(?:不清|模糊|欠清)')
EFFACE = re.compile(r'脑沟[^，,。；]{0,6}(?:变浅|消失|受压)|脑池[^，,。；]{0,6}(?:受压|变窄|闭塞|消失)|脑室[^，,。；]{0,8}(?:变窄|变小|稍小|受压|缩小|显示不清)'
                    r'|外侧裂[^，,。；]{0,4}(?:闭塞|变窄)')
SUBCORT = re.compile(r'皮层下|皮质下|U形纤维|U纤维|皮髓质交界|灰白质交界')
LOBAR_WM = re.compile(r'(?:额|顶|颞|枕)叶?[^，,。；]{0,6}白质|大脑半球[^，,。；]{0,4}白质')
EXTENSIVE = re.compile(r'双侧|两侧|多发|弥漫|广泛|大脑半球|额顶|顶枕|额颞|颞顶|颞枕')
UNSURE = re.compile(r'似|可疑|[？?]')
HEMI = re.compile(r'(?<!双侧)(?<!两侧)(?:左|右)(?:侧)?大脑半球')
CHRONIC = re.compile(r'萎缩|软化|囊变|胶质增生|陈旧')


def clauses(text):
    for sent in SENT.split(str(text)):
        if sent.strip():
            yield sent, [c for c in CLAUSE.split(sent) if c.strip()]


def positive_clauses(text, region):
    # Clauses naming the region together with an abnormal descriptor and no negation.
    return [(sent, c) for sent, cls in clauses(text) for c in cls if re.search(region, c) and ABN.search(c) and not NEG_IMG.search(c)]


def thalamic_side(clause):
    # Nearest side marker before "丘脑" in its list item; otherwise inherited from the nearest preceding list item.
    items = clause.split('、')
    for i, it in enumerate(items):
        p = it.find('丘脑')
        if p < 0:
            continue
        marks = SIDE.findall(it[:p])
        if not marks:
            if '对称' in it[p:]:
                return 'bilateral'
            for prev in reversed(items[:i]):
                mm = SIDE.findall(prev)
                if mm:
                    marks = [mm[0]]
                    break
        if not marks:
            return 'unspecified'
        return 'unilateral' if marks[-1] in ('左', '右') else 'bilateral'
    return None


def find_thalamic(text, bilateral):
    for sent, c in positive_clauses(text, r'丘脑'):
        side = thalamic_side(c)
        if side is None:
            continue
        if side == 'unspecified' and re.search(r'对称|双侧丘脑|两侧丘脑', sent):
            side = 'bilateral'
        if not bilateral or side == 'bilateral':
            return c.strip()
    return ''


def find_splenial(text):
    for sent, cls in clauses(text):
        if '胼胝体' in sent and '压部' in sent:
            for c in cls:
                if '压部' in c and (ABN.search(c) or DWI_POS.search(c)) and not NEG_IMG.search(c):
                    return sent.strip()
    m = re.search(r'[^。；]*(?:可逆性胼胝体压部|胼胝体压部[^，,。；]{0,8}(?:综合征|脑病|病变))[^。；]*', str(text))
    if m and not re.search(r'未见|消失|吸收', m.group(0)):
        return m.group(0).strip()
    return ''


def find_subcortical_dwi(text, region=SUBCORT):
    # Extensive (bilateral or multilobar) white-matter lesions with restricted diffusion in the same sentence (bright-tree-type pattern).
    for sent, cls in clauses(text):
        if not region.search(sent) or CHRONIC.search(sent) or UNSURE.search(sent) or not EXTENSIVE.search(sent):
            continue
        reg = any(region.search(c) and (ABN.search(c) or DWI_POS.search(c)) and not NEG_IMG.search(c) for c in cls)
        dwi = any(DWI_POS.search(c) and not NEG_IMG.search(c) for c in cls)
        if reg and dwi:
            return sent.strip()
    return ''


def find_diffuse_edema(text):
    # Global swelling, herniation or cisternal effacement; bilateral loss of gray-white differentiation counts only with effacement.
    efface = any(EFFACE.search(c) and not re.search(r'未见|无明显|无|未', c) for _, cls in clauses(text) for c in cls)
    for sent, cls in clauses(text):
        for c in cls:
            m = DIFFUSE_EDEMA.search(c)
            if m and not re.search(r'未见|无明显|无|未|排除', c[max(0, m.start() - 8):m.start()]):
                return sent.strip()
        m = GWM_LOSS.search(sent)
        if m and efface and not re.search(r'未见|无明显|无|未', sent[max(0, m.start() - 8):m.start()]):
            return sent.strip()
    return ''


def find_hemispheric(text):
    for sent, cls in clauses(text):
        if re.search(r'双侧大脑半球|两侧大脑半球', sent) or CHRONIC.search(sent):
            continue
        if HEMI.search(sent) and (re.search(r'肿胀|水肿', sent) or DWI_POS.search(sent)) and not NEG_IMG.search(sent):
            return sent.strip()
    return ''


def imaging_table(sheet, head_only):
    # The supplement holds findings + conclusion; the main export holds the report name. Merge by admission and time.
    s = supp[sheet].copy(); s['vis'] = s['科研就诊编号'].astype(str); s['time'] = pd.to_datetime(s['检查时间'], errors='coerce')
    s['text'] = s['检查所见'].fillna('').astype(str) + '。' + s['检查结论'].fillna('').astype(str)
    s['concl'] = s['检查结论'].fillna('').astype(str)
    m = sheets[sheet].copy(); m['vis'] = m['科研就诊编号'].astype(str); m['time'] = pd.to_datetime(m['检查时间'], errors='coerce')
    m['name'] = m['报告名称'].fillna('').astype(str) if '报告名称' in m else ''
    m['concl_main'] = m['检查结论'].fillna('').astype(str)
    x = s.merge(m[['vis', 'time', 'name']].drop_duplicates(['vis', 'time']), on=['vis', 'time'], how='left')
    only_main = m[~m.set_index(['vis', 'time']).index.isin(x.set_index(['vis', 'time']).index)].copy()
    only_main['text'] = only_main['concl_main']; only_main['concl'] = only_main['concl_main']
    x = pd.concat([x[['vis', 'time', 'name', 'text', 'concl']], only_main[['vis', 'time', 'name', 'text', 'concl']]], ignore_index=True)
    x['name'] = x['name'].fillna('')
    x = x.merge(idx[['vis', 'admit', 'discharge']], on='vis')
    x = x[(x.time >= x.admit - pd.Timedelta(days=1)) & (x.time <= x.discharge + pd.Timedelta(days=1))].copy()
    if head_only:
        x = x[x.name.str.contains(r'头|颅|脑', regex=True) | (x.name == '') & x.text.str.contains(r'脑室|脑沟|基底节|丘脑|小脑|中线结构')]
    x['day'] = (x.time - x.admit).dt.total_seconds() / 86400
    return x.sort_values(['vis', 'time']).reset_index(drop=True)


MRI = imaging_table('MRI报告', head_only=False)
MRI = MRI[MRI.text.str.contains(r'脑|丘脑|胼胝体|基底节|小脑|中线结构')].copy()
CT = imaging_table('CT报告', head_only=True)
FEATURES = {'thal_bilat': lambda t: find_thalamic(t, True), 'thal_any': lambda t: find_thalamic(t, False),
            'diffuse_edema': find_diffuse_edema, 'hemispheric': find_hemispheric}
for X in (MRI, CT):
    for k, f in FEATURES.items():
        X[k + '_txt'] = X.text.apply(f)
        X[k] = X[k + '_txt'].ne('')
    X['brainstem'] = X.text.apply(lambda t: bool(positive_clauses(t, r'脑干|桥脑|脑桥|延髓|中脑|大脑脚')))
for k, f in {'splenial': find_splenial, 'subcortical_dwi': find_subcortical_dwi, 'lobar_wm_dwi': lambda t: find_subcortical_dwi(t, LOBAR_WM)}.items():
    MRI[k + '_txt'] = MRI.text.apply(f)
    MRI[k] = MRI[k + '_txt'].ne('')
MRI['any_dwi'] = MRI.text.apply(lambda t: any(DWI_POS.search(c) and not NEG_IMG.search(c) for _, cls in clauses(t) for c in cls))
MRI['normal'] = MRI.concl.str.contains(r'未见明显异常|未见异常') & ~MRI[['thal_any', 'splenial', 'subcortical_dwi', 'diffuse_edema', 'any_dwi']].any(axis=1)
MRI['imp_ane'] = MRI.concl.str.contains(r'坏死性脑病') & ~MRI.concl.str.contains(r'不支持|排除|除外')
MRI['imp_mers'] = MRI.concl.str.contains(r'可逆性胼胝体|胼胝体压部[^，,。；]{0,8}(?:综合征|脑病)')
MRI['imp_aesd'] = MRI.concl.str.contains(r'双相|双向|迟发性弥散|后期弥散|AESD')
MRI['imp_ane_txt'] = np.where(MRI.imp_ane, MRI.concl.str.replace(r'\s+', ' ', regex=True).str.slice(0, 120), '')


def per_admission(X, prefix, flags):
    out = X.groupby('vis').agg(**{f'{prefix}_n': ('text', 'size'), f'{prefix}_first_day': ('day', 'min')},
                               **{f'{prefix}_{c}': (c, 'any') for c in flags})
    for c in flags:
        if c + '_txt' in X:
            pos = X[X[c]]
            out[f'{prefix}_{c}_day'] = pos.groupby('vis').day.min()
            out[f'{prefix}_{c}_evidence'] = pos.groupby('vis').apply(lambda g: f"{prefix.upper()} day {g.day.iloc[0]:.0f}: {g[c + '_txt'].iloc[0][:160]}")
    return out


mri_v = per_admission(MRI, 'mri', ['thal_bilat', 'thal_any', 'brainstem', 'diffuse_edema', 'hemispheric', 'splenial', 'subcortical_dwi',
                                    'lobar_wm_dwi', 'any_dwi', 'normal', 'imp_ane', 'imp_mers', 'imp_aesd'])
for c in ['subcortical_dwi', 'lobar_wm_dwi']:
    mri_v[f'mri_{c}_day2plus'] = MRI[MRI.day >= 2].groupby('vis')[c].any()
    mri_v[f'mri_{c}_day2plus'] = mri_v[f'mri_{c}_day2plus'].fillna(False).astype(bool)
ct_v = per_admission(CT, 'ct', ['thal_bilat', 'thal_any', 'diffuse_edema', 'hemispheric'])
print('Index admissions with brain MRI:', mri_v.shape[0], '| with head CT:', ct_v.shape[0])
print({k: int(mri_v[k].sum()) for k in mri_v.columns if mri_v[k].dtype == bool})
print({k: int(ct_v[k].sum()) for k in ct_v.columns if ct_v[k].dtype == bool})

In [ ]:
# ITES evidence 2/4: EEG, CSF, fever, altered consciousness, seizures.
eeg = supp['脑电图检查'].copy(); eeg['vis'] = eeg['科研就诊编号'].astype(str)
eeg['text'] = (eeg['背景活动'].fillna('').astype(str) + '。' + eeg['印象'].fillna('').astype(str))
eeg = eeg[eeg.vis.isin(set(idx.vis))]
EEG_SLOW = re.compile(r'慢波活动|弥漫性慢波|广泛性慢波|弥漫性\s*\d|广泛性\s*\d|背景[^，,。；]{0,10}慢|慢化|δ波|δ活动|θ活动|θ波[^，,。；]{0,4}(?:增多|为主)'
                      r'|慢波为主|慢波增多|慢波较多|大量慢波|多量慢波|慢波明显')
EEG_SUPP = re.compile(r'电静息|暴发[-—]?抑制|爆发[-—]?抑制|低平抑制|抑制性脑波|低电压|电压低平|背景抑制')
EEG_SZ = re.compile(r'癫痫持续状态|电持续状态|非惊厥|NCSE|发作期')
eeg['slow'] = eeg.text.apply(lambda t: any(EEG_SLOW.search(c) and not re.search(r'未见|少量|无明显', c) for _, cls in clauses(t) for c in cls))
eeg['suppressed'] = eeg.text.str.contains(EEG_SUPP)
eeg['seizure_pattern'] = (eeg['印象'].fillna('').astype(str).str.contains(EEG_SZ)) | \
                         (eeg['发作期'].fillna('').astype(str).str.contains(r'监测到|发作') & ~eeg['发作期'].fillna('').astype(str).str.contains(r'未见特殊|未发生'))
eeg_v = eeg.groupby('vis').agg(eeg_n=('text', 'size'), eeg_slow=('slow', 'any'), eeg_suppressed=('suppressed', 'any'), eeg_seizure=('seizure_pattern', 'any'))

csf = pd.concat([sheets['实验室检查_脑脊液常规'], supp['实验室检查_脑脊液常规']]).drop_duplicates('报告编号')
csf['vis'] = csf['就诊编号'].astype(str); csf['wbc'] = parse_lab_value(csf['白细胞计数定量-定量结果'], qual_col(csf, '白细胞计数定量-定量结果'))[0]
csf_v = csf[csf.vis.isin(set(idx.vis))].groupby('vis').agg(csf_wbc_max=('wbc', 'max'))
cbio = pd.concat([sheets['实验室检查_脑脊液生化'], supp['实验室检查_脑脊液生化']]).drop_duplicates('报告编号')
cbio['vis'] = cbio['就诊编号'].astype(str)
prot = parse_lab_value(cbio['脑脊液蛋白定量-定量结果'], qual_col(cbio, '脑脊液蛋白定量-定量结果'))[0]
unit = cbio['脑脊液蛋白定量-定量结果单位'].astype(str).str.lower()
cbio['protein_gL'] = np.where(unit.str.contains('mg'), prot / 1000, prot)
csf_v = csf_v.join(cbio[cbio.vis.isin(set(idx.vis))].groupby('vis').protein_gL.max().rename('csf_protein_max'), how='outer')
vd = supp['实验室检查_脑脊液病毒DNA测定'].copy(); vd['vis'] = vd['就诊编号'].astype(str)
qual = [c for c in vd.columns if c.endswith('-定性结果')]
vd['positive'] = vd[qual].astype(str).apply(lambda r: r.str.contains('阳').any(), axis=1)
csf_v = csf_v.join(vd[vd.vis.isin(set(idx.vis))].groupby('vis').positive.any().rename('csf_virus_positive'), how='outer')

an = supp['儿科入院记录'].copy(); an['vis'] = an['科研就诊编号'].astype(str); an = an[an.vis.isin(set(idx.vis))]
an = an.groupby('vis').agg(cc=('主诉', lambda s: ' '.join(map(str, s.dropna()))), hpi=('现病史', lambda s: ' '.join(map(str, s.dropna()))),
                           pmh=('既往史', lambda s: ' '.join(map(str, s.dropna()))), pe=('体格检查', lambda s: ' '.join(map(str, s.dropna()))))
FEVER = re.compile(r'发热|发烧|高热|低热|中热|热峰|体温[^，,。；]{0,6}(?:升高|最高|达3[89]|达4)')
ENC = re.compile(r'昏迷|嗜睡|昏睡|意识障碍|意识不清|意识模糊|神志不清|神志模糊|神志恍惚|神志昏睡|神志嗜睡|神志浅昏迷|呼之不应|反应差|反应迟钝'
                 r'|反应低下|精神萎靡|精神反应差|精神差|谵妄|胡言乱语|烦躁不安|易激惹|GCS|格拉斯哥|淡漠|不认人|行为异常|精神行为异常|意识改变')
SEIZ = re.compile(r'抽搐|惊厥|抽动')
NEG_TXT = re.compile(r'无|不伴|否认|未|没有|不详')


def mentions(text, pat):
    return any(pat.search(c) and not NEG_TXT.search(c[:max(0, pat.search(c).start())][-6:]) for _, cls in clauses(text) for c in cls)


ev = idx[['vis', 'patient', 'subtype', 'age_y', 'admit', 'discharge', 'icu', 'death', 'poor', 'dx_text', 'course']].copy()
ev = ev.merge(an, left_on='vis', right_index=True, how='left')
for c in ['cc', 'hpi', 'pmh', 'pe']:
    ev[c] = ev[c].fillna('')
vit = supp['患者生命体征'].copy(); vit['vis'] = vit['科研就诊编号'].astype(str); vit = vit[vit.vis.isin(set(idx.vis))]
vit['T'] = pd.to_numeric(vit['体温(°C)'], errors='coerce'); vit['t'] = pd.to_datetime(vit['测量时间'], errors='coerce')
vit = vit.merge(idx[['vis', 'admit']], on='vis')
early = vit[(vit.t - vit.admit).dt.total_seconds().between(-86400, 3 * 86400)]
ev['tmax_72h'] = ev.vis.map(early.groupby('vis')['T'].max())
ev['fever_text'] = [mentions(a + '。' + b, FEVER) for a, b in zip(ev.cc, ev.hpi)]
ev['fever_course'] = ev.course.apply(lambda t: mentions(t[:400], FEVER))
ev['fever'] = ev.fever_text | ev.fever_course | (ev.tmax_72h >= 38.0)
ev['encephalopathy_text'] = [mentions(a + '。' + b + '。' + c, ENC) or mentions(d[:600], ENC) for a, b, c, d in zip(ev.cc, ev.hpi, ev.pe, ev.course)]
ev['seizure_text'] = [mentions(a + '。' + b, SEIZ) or mentions(c[:600], SEIZ) for a, b, c in zip(ev.cc, ev.hpi, ev.course)]
ev['biphasic_text'] = (ev.hpi + ev.course).str.contains(r'再次抽搐|再发抽搐|再次出现抽搐|又出现抽搐|再次惊厥|再次发作|双相|双向|第二次抽搐')
ev['prior_epilepsy'] = ev.pmh.apply(lambda t: mentions(t, re.compile(r'癫痫(?!持续)|婴儿痉挛'))) | \
                       ev.hpi.str.contains(r'诊断[“"]?癫痫|癫痫病史|既往.{0,8}癫痫')
ev['prior_neuro_disease'] = ev.pmh.apply(lambda t: mentions(t, re.compile(r'脑瘫|发育落后|发育迟缓|智力低下|遗传代谢|线粒体|脑积水')))
for t in (mri_v, ct_v, eeg_v, csf_v):
    ev = ev.merge(t, left_on='vis', right_index=True, how='left')
print('Fever documented:', int(ev.fever.sum()), '| altered consciousness documented:', int(ev.encephalopathy_text.sum()),
      '| EEG available:', int(ev.eeg_n.notna().sum()), '| CSF WBC available:', int(ev.csf_wbc_max.notna().sum()))

In [ ]:
# ITES evidence 3/4: shock and the six multi-organ items used for ASEM (first 72 h of admission).
def first72(sheet, col, how):
    parts = []
    for book in (sheets, supp):
        if sheet in book and col + '-定量结果' in book[sheet].columns:
            d = book[sheet]
            parts.append(pd.DataFrame({'report': d['报告编号'].astype(str), 'vis': d['就诊编号'].astype(str),
                                       'v': parse_lab_value(d[col + '-定量结果'], qual_col(d, col + '-定量结果'))[0],
                                       'uln': pd.to_numeric(d.get(col + '-正常高值'), errors='coerce'),
                                       'lln': pd.to_numeric(d.get(col + '-正常低值'), errors='coerce'),
                                       't': pd.to_datetime(d['报告时间'], errors='coerce')}))
    x = pd.concat(parts).drop_duplicates('report').dropna(subset=['v'])
    x = x.merge(idx[['vis', 'admit']], on='vis')
    x = x[(x.t - x.admit).dt.total_seconds().between(-12 * 3600, 72 * 3600)]
    return x.groupby('vis').apply(how)


ev['hb_min'] = ev.vis.map(first72('实验室检查_血常规', '血红蛋白浓度定量', lambda g: g.v.min()))
ev['plt_min'] = ev.vis.map(first72('实验室检查_血常规', '血小板计数定量', lambda g: g.v.min()))
ev['inr_max'] = ev.vis.map(first72('实验室检查_凝血功能检查', '国际标准比率定量', lambda g: g.v.max()))
ev['fib_low'] = ev.vis.map(first72('实验室检查_凝血功能检查', '纤维蛋白原检测定量', lambda g: bool((g.v < g.lln).any())))
ev['ph_min'] = ev.vis.map(first72('实验室检查_血气分析', '酸碱度定量', lambda g: g.v.min()))
ev['be_min'] = ev.vis.map(first72('实验室检查_血气分析', '碱剩余定量', lambda g: g.v.min()))
ev['alt_x_uln'] = ev.vis.map(first72('实验室检查_血液肝功能测定', '谷丙转氨酶定量', lambda g: (g.v / g.uln).max()))
ev['ast_x_uln'] = ev.vis.map(first72('实验室检查_血液肝功能测定', '谷草转氨酶定量', lambda g: (g.v / g.uln).max()))
ev['cr_x_uln'] = ev.vis.map(first72('实验室检查_血液肾功能测定', '肌酐定量', lambda g: (g.v / g.uln).max()))
ev['ammonia_max'] = ev.vis.map(first72('实验室检查_血氨AMMO', '血氨AMMO定量', lambda g: g.v.max()))
ev['na_max'] = ev.vis.map(first72('实验室检查_血液电解质测定', '钠定量', lambda g: g.v.max()))
dx = ev.dx_text.fillna('')
MOF = {
    'anemia': ev.hb_min < 100,                                                         # Hb < 100 g/L
    'thrombocytopenia': ev.plt_min < 100,                                              # platelets < 100 x 10^9/L
    'dic': ((ev.inr_max >= 1.5) & ev.fib_low.fillna(False).astype(bool)) | dx.str.contains(r'弥散性血管内凝血|DIC'),
    'acidosis': (ev.ph_min < 7.30) | (ev.be_min <= -8),
    'liver': (ev.alt_x_uln > 2) | (ev.ast_x_uln > 2),                                   # > 2x age-specific ULN
    'renal': (ev.cr_x_uln > 1) | dx.str.contains(r'急性肾损伤|急性肾衰|急性肾功能'),        # > age-specific ULN
}
for k, v in MOF.items():
    ev[f'mof_{k}'] = v.fillna(False).astype(bool)
ev['mof_count'] = ev[[f'mof_{k}' for k in MOF]].sum(axis=1)
vaso72 = idx.set_index('vis').vaso_time.sub(idx.set_index('vis').admit).dt.total_seconds().le(72 * 3600)
ev['shock'] = ev.vis.map(vaso72).fillna(False).astype(bool) | dx.str.contains(r'休克') & ~dx.str.contains(r'失血性休克|心源性休克')
ev['septic_shock'] = dx.str.contains(r'脓毒性休克|感染性休克|脓毒症休克')
ev['sepsis_dx'] = dx.str.contains(r'脓毒症|败血症|脓毒性休克|感染性休克')
ev['cpr'] = ev.vis.map(idx.set_index('vis').cpr_any).fillna(False).astype(bool)
print('MOF items:', {k: int(ev[f'mof_{k}'].sum()) for k in MOF}, '| shock:', int(ev.shock.sum()), '| septic shock dx:', int(ev.septic_shock.sum()))

In [ ]:
# ITES evidence 4/4: alternative diagnoses that the consensus requires to be excluded.
EXCLUSIONS = {   # category: (pattern on discharge diagnoses of the index admission, applies to imaging-defined subtypes too)
    'CNS infection (bacterial/TB/fungal) or specific viral encephalitis': (r'化脓性脑膜炎|细菌性脑膜炎|结核性脑膜炎|隐球菌|真菌性脑膜炎|脑脓肿|疱疹病毒性脑炎|单纯疱疹病毒性脑炎|单纯疱疹性脑炎|HSV脑炎|乙型脑炎|流行性乙型', True),
    'Autoimmune or demyelinating disease': (r'自身免疫性脑炎|自体免疫性脑炎|免疫介导性脑炎|NMDA|GABA|LGI1|MOG|ADEM|急性播散性脑脊髓炎|视神经脊髓炎|多发性硬化|桥本脑病', True),
    'Metabolic disease, Reye syndrome or IEM': (r'瑞氏|Reye|瑞夷|线粒体|Leigh|亚急性坏死性|甲基丙二酸|戊二酸|有机酸血症|尿素循环|遗传代谢|先天性代谢|代谢性脑病'
                                              r'|低血糖脑病|低血糖昏迷|肝性脑病|尿毒症|Wernicke|韦尼克|维生素B1缺乏', True),
    'Hypoxic-ischemic injury, drowning or asphyxia': (r'溺水|窒息|缺氧缺血性脑病|缺血缺氧性脑病|缺氧性脑病|缺氧缺血性脑损伤', True),
    'Trauma': (r'颅脑损伤|颅脑外伤|脑挫|硬膜外血肿|外伤性|颅骨骨折|虐待', True),
    'Poisoning': (r'一氧化碳|药物中毒|农药|有机磷|毒蕈|蘑菇中毒|鼠药|酒精中毒|乙醇中毒|铅中毒|汞中毒|百草枯|药物过量', True),
    'Cerebrovascular disease or tumor': (r'脑梗|静脉窦血栓|烟雾病|颅内动脉瘤|动静脉畸形|血管炎|脑肿瘤|母细胞瘤|胶质瘤|室管膜瘤|生殖细胞瘤|颅内占位', True),
    'PRES or hypertensive encephalopathy': (r'可逆性后部|PRES|高血压脑病', True),
    'Systemic inflammatory disease (HLH, Kawasaki disease, MIS-C, SLE)': (r'噬血|HLH|巨噬细胞活化|川崎|多系统炎症综合征|MIS-C|系统性红斑狼疮|狼疮', False),
    'Epilepsy or developmental and epileptic encephalopathy': (r'癫痫(?!持续)|婴儿痉挛|West综合征|Dravet|Lennox', False),
    'Malignancy or chemotherapy': (r'白血病|淋巴瘤|神经母细胞瘤|化疗', False),
}
UNCERTAIN = re.compile(r'[？?]|待排|待查|可能|不除外|疑似|可疑')
dx_items = ev.dx_text.fillna('').str.split(r'\s*\|\s*')


def excl_status(items, pat):
    # 'definite' when an item names the diagnosis without an uncertainty marker; 'possible' when only qualified mentions exist.
    hits = [x for x in items if re.search(pat, x)]
    if not hits:
        return ''
    return 'definite' if any(not UNCERTAIN.search(x) for x in hits) else 'possible'


for k, (p, _) in EXCLUSIONS.items():
    st_ = dx_items.apply(lambda L: excl_status(L, p))
    ev['excl: ' + k] = st_.eq('definite')
    ev['possible excl: ' + k] = st_.eq('possible')
ev['excl: Epilepsy or developmental and epileptic encephalopathy'] |= ev.prior_epilepsy
FIRES_LABEL = ev.subtype.eq('FIRES') | dx.str.contains(r'FIRES|热性感染相关|发热感染相关|感染相关性癫痫综合征')
ev.loc[FIRES_LABEL, 'excl: Epilepsy or developmental and epileptic encephalopathy'] = False   # epilepsy is the FIRES phenotype itself
ev['neonate'] = ev.age_y < 28 / 365
ev['csf_pleocytosis'] = ev.csf_wbc_max > 20
ev['csf_virus_positive'] = ev.csf_virus_positive.fillna(False).astype(bool)
HARD = ['excl: ' + k for k, (_, hard) in EXCLUSIONS.items() if hard]
SOFT = ['excl: ' + k for k, (_, hard) in EXCLUSIONS.items() if not hard]
print('Exclusion flags:', {k.replace('excl: ', ''): int(ev[k].sum()) for k in HARD + SOFT})
print('CSF WBC > 20 x10^6/L:', int(ev.csf_pleocytosis.sum()), '| CSF virus PCR positive:', int(ev.csf_virus_positive.sum()), '| neonates:', int(ev.neonate.sum()))

In [ ]:
# ITES assignment (hierarchical; every step is recorded in `ites_reason`).
def b(row, col):
    v = row.get(col)
    return bool(v) if pd.notna(v) else False


def assign(row):
    hard = [c.replace('excl: ', '') for c in HARD if row[c]]
    soft = [c.replace('excl: ', '') for c in SOFT if row[c]]
    thal = b(row, 'mri_thal_bilat') or b(row, 'ct_thal_bilat') or b(row, 'mri_imp_ane')
    thal_day = np.nanmin([row.get('mri_thal_bilat_day', np.nan), row.get('ct_thal_bilat_day', np.nan), np.nan if not b(row, 'mri_imp_ane') else row.get('mri_first_day', np.nan)])
    edema = b(row, 'mri_diffuse_edema') or b(row, 'ct_diffuse_edema')
    mers_img = b(row, 'mri_splenial') or b(row, 'mri_imp_mers')
    aesd_img = b(row, 'mri_subcortical_dwi') or b(row, 'mri_imp_aesd') or (b(row, 'mri_lobar_wm_dwi_day2plus') and not mers_img) or \
        (row.subtype == 'AESD' and b(row, 'mri_any_dwi'))
    hemi = b(row, 'mri_hemispheric') or b(row, 'ct_hemispheric')
    has_neuroimaging = pd.notna(row.get('mri_n')) or pd.notna(row.get('ct_n'))
    fires = row.subtype == 'FIRES' or FIRES_LABEL.loc[row.name]
    flag = '' if row.fever else ' [fever not documented]'
    if row.neonate:
        return 'Non-ITES: neonatal encephalopathy', 'Neonate (<28 days)'
    if hard:
        return 'Non-ITES: alternative diagnosis', '; '.join(hard)
    if row.csf_virus_positive or row.csf_pleocytosis:
        return 'Non-ITES: encephalitis/meningitis', f'CSF WBC max {row.csf_wbc_max:g} x10^6/L' if row.csf_pleocytosis else 'CSF viral PCR positive'
    if thal and not (fires and not thal_day <= 7):
        return 'ANE', ('Bilateral thalamic lesions' if (b(row, 'mri_thal_bilat') or b(row, 'ct_thal_bilat')) else 'Radiologist impression of ANE') + flag
    if fires:
        return 'FIRES', 'Clinician-diagnosed FIRES' + flag
    if edema:
        if row.septic_shock:
            return 'SAE', 'Diffuse cerebral edema with septic shock (septic shock excludes AFCE/ASEM)'
        if row.shock and row.mof_count >= 3:
            return 'ASEM', f'Diffuse cerebral edema, shock, {int(row.mof_count)} of 6 organ items' + flag
        return 'AFCE', 'Diffuse cerebral edema' + flag
    if aesd_img:
        late = b(row, 'mri_subcortical_dwi_day2plus') or b(row, 'mri_lobar_wm_dwi_day2plus')
        return 'AESD', 'White-matter restricted diffusion' + (' (MRI on day >= 2)' if late else ' (day of MRI < 2: check timing)') + flag
    if mers_img:
        if 'Systemic inflammatory disease (HLH, Kawasaki disease, MIS-C, SLE)' in soft:
            return 'Non-ITES: alternative diagnosis', 'Splenial lesion with systemic inflammatory disease (MERS exclusion)'
        return 'MERS', 'Splenial lesion' + (' with restricted diffusion' if b(row, 'mri_any_dwi') else ' (diffusion not stated)') + flag
    if hemi and row.seizure_text:
        return 'HHE', 'Unilateral hemispheric swelling after seizures' + flag
    if row.subtype in ('ANE', 'MERS', 'AESD') and not has_neuroimaging:
        return row.subtype, 'Clinician-diagnosed; no neuroimaging report in the export' + flag
    if row.septic_shock or row.subtype == 'Septic_encephalopathy':
        return 'SAE', 'Septic shock or clinician-labeled septic encephalopathy, no ITES-specific imaging'
    if soft:
        return 'Non-ITES: alternative diagnosis', '; '.join(soft)
    if not row.fever:
        return 'Unclassifiable', 'No documented febrile illness'
    eeg_enceph = b(row, 'eeg_slow') or b(row, 'eeg_suppressed')
    if eeg_enceph and row.encephalopathy_text:
        return 'Possible ITES', 'Fever, altered consciousness, encephalopathic EEG, no syndrome-specific imaging'
    if eeg_enceph:
        return 'Unclassifiable', 'Fever and encephalopathic EEG, but altered consciousness not documented; no syndrome-specific imaging'
    if pd.isna(row.get('eeg_n')) and not has_neuroimaging:
        return 'Unclassifiable', 'No EEG and no neuroimaging'
    return 'Unclassifiable', 'No syndrome-specific imaging and no encephalopathic EEG documented'


ev[['ites_class', 'ites_reason']] = ev.apply(lambda r: pd.Series(assign(r)), axis=1)
TIER = {'ANE': 'Definite ITES', 'ASEM': 'Definite ITES', 'AFCE': 'Definite ITES', 'AESD': 'Definite ITES', 'MERS': 'Definite ITES',
        'HHE': 'ITES-related', 'FIRES': 'ITES-related', 'Possible ITES': 'Possible ITES', 'SAE': 'Sepsis-associated encephalopathy',
        'Unclassifiable': 'Unclassifiable'}
ev['ites_tier'] = ev.ites_class.map(TIER).fillna('Non-ITES')
ev['ites_algorithmic'] = ev.ites_class
CLIN = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'AESD', 'FIRES': 'FIRES', 'Septic_encephalopathy': 'SAE'}
ev['clinician_label'] = ev.subtype.map(CLIN).fillna('Other/unspecified')
specific_img = ev.ites_reason.str.contains(r'thalamic|edema|white-matter|Splenial|hemispheric', case=False)
conflict_excl = ev.ites_class.str.startswith('Non-ITES') & (ev.clinician_label.isin(['ANE', 'MERS', 'AESD', 'FIRES']) |
                 (ev.mri_thal_bilat.fillna(False).astype(bool) | ev.mri_splenial.fillna(False).astype(bool)))
possible_excl = ev[[c for c in ev.columns if c.startswith('possible excl: ')]].any(axis=1)
ev['review_reasons'] = [
    '; '.join(r for r, f in [
        ('clinician label differs from algorithm', cl not in ('Other/unspecified',) and cl != ic),
        ('exclusion conflicts with ITES-type imaging or clinician ITES label', ce),
        ('fever not documented', 'fever not documented' in rs),
        ('AESD imaging before day 2', 'check timing' in rs),
        ('splenial lesion without stated restricted diffusion', 'diffusion not stated' in rs),
        ('clinician label without neuroimaging in export', 'no neuroimaging report' in rs),
        ('qualified (?) alternative diagnosis present', pe and not ic.startswith('Non-ITES')),
        ('cardiac arrest/CPR documented', cp and ic in ('ANE', 'ASEM', 'AFCE', 'AESD')),
        ('altered consciousness not documented', (not en) and ic in ('ANE', 'ASEM', 'AFCE', 'AESD', 'MERS', 'HHE', 'FIRES')),
    ] if f)
    for cl, ic, ce, rs, pe, cp, en in zip(ev.clinician_label, ev.ites_class, conflict_excl, ev.ites_reason, possible_excl, ev.cpr, ev.encephalopathy_text)]
ev['needs_review'] = ev.review_reasons.ne('')
ADJ = HERE / 'output_ites' / 'ites_adjudicated.csv'
if ADJ.exists():
    adj = pd.read_csv(ADJ, dtype={'vis': str}).dropna(subset=['adjudicated_class'])
    ev = ev.merge(adj[['vis', 'adjudicated_class']], on='vis', how='left')
    ev['ites_class'] = ev.adjudicated_class.fillna(ev.ites_class)
    ev['ites_tier'] = ev.ites_class.map(TIER).fillna('Non-ITES')
    print('Adjudicated categories applied:', int(ev.adjudicated_class.notna().sum()))
else:
    print('No adjudication file found: algorithmic categories are used.')
print(ev.ites_class.value_counts().to_string())

In [ ]:
# Operational criteria (English summary of every rule used above; exported for the supplement).
CRITERIA = pd.DataFrame([
    ('Population', 'Index admission', 'First acute-phase admission per patient with an acute encephalopathy diagnosis and a recorded discharge disposition.'),
    ('Core criterion', 'Febrile illness', 'Fever in the chief complaint or history of present illness (clause-level negation), in the first 400 characters of the hospital course, or a recorded temperature ≥ 38.0°C from 24 h before to 72 h after admission. Required for possible ITES; for syndromes defined by neuroimaging or FIRES, absence is flagged for adjudication rather than used to exclude, because the consensus criterion also accepts infection without fever and documentation is incomplete in retrospective records.'),
    ('Core criterion', 'Altered consciousness', 'Coma, stupor, somnolence, lethargy, confusion, delirium, irritability or GCS documentation in the admission note or hospital course (clause-level negation). Required for possible ITES; for syndromes defined by neuroimaging or FIRES, absence is flagged for adjudication.'),
    ('Exclusion', 'Neonate', 'Age < 28 days.'),
    ('Exclusion', 'Alternative diagnoses', 'Unqualified discharge diagnosis of: bacterial/tuberculous/fungal CNS infection or HSV/Japanese encephalitis; autoimmune encephalitis, ADEM or other demyelinating disease; metabolic encephalopathy, inborn error of metabolism, Reye syndrome, hepatic or uremic encephalopathy, HUS; hypoxic-ischemic injury, drowning or asphyxia; trauma; poisoning; cerebral infarction, venous sinus thrombosis, vasculitis or tumor; PRES/hypertensive encephalopathy. Diagnoses qualified by "?" or "possible" are flagged for review only.'),
    ('Exclusion', 'Encephalitis by CSF', 'CSF white-cell count > 20 × 10^6/L at any lumbar puncture during the admission, or a positive CSF viral PCR.'),
    ('Exclusion (soft)', 'Systemic inflammatory disease, epilepsy, malignancy', 'HLH/MAS, Kawasaki disease, MIS-C or SLE; known epilepsy or developmental and epileptic encephalopathy (except FIRES); leukemia/lymphoma/chemotherapy. Applied only when no syndrome-specific neuroimaging is present (Kawasaki disease also excludes MERS).'),
    ('Subtype', 'ANE', 'Bilateral thalamic lesions on MRI or CT (side-aware parsing of the report), or a radiologist impression of acute necrotizing encephalopathy.'),
    ('Subtype', 'FIRES', 'Clinician diagnosis of FIRES; takes precedence over thalamic lesions first seen after day 7.'),
    ('Subtype', 'ASEM', 'Diffuse cerebral edema plus shock (vasoactive infusion within 72 h or shock diagnosis, not hemorrhagic/cardiogenic) plus ≥ 3 of: Hb < 100 g/L; platelets < 100 × 10^9/L; DIC (INR ≥ 1.5 with fibrinogen below the reference limit, or DIC diagnosis); acidosis (pH < 7.30 or base excess ≤ −8 mmol/L); ALT or AST > 2x age-specific upper limit; creatinine above the age-specific upper limit or AKI diagnosis (first 72 h).'),
    ('Subtype', 'AFCE', 'Diffuse cerebral edema (global swelling, tonsillar herniation, cisternal effacement, or bilateral loss of gray-white differentiation with effacement) not meeting ASEM; septic shock diagnosis reassigns to SAE.'),
    ('Subtype', 'AESD', 'Bilateral or multilobar subcortical white-matter restricted diffusion (bright-tree-type), lobar white-matter restricted diffusion on MRI from day 2 without a splenial lesion, or clinician-diagnosed AESD with any restricted diffusion.'),
    ('Subtype', 'MERS', 'Splenial lesion of the corpus callosum on MRI (or radiologist impression of MERS/RESLES); restricted diffusion recorded when stated.'),
    ('Subtype', 'HHE', 'Unilateral hemispheric swelling or restricted diffusion after seizures.'),
    ('Non-ITES comparator', 'SAE', 'Septic shock diagnosis or clinician-labeled septic encephalopathy without ANE, FIRES, AESD, MERS, or HHE features; children with diffuse cerebral edema and septic shock are also classified as SAE because septic shock is an exclusion for AFCE and ASEM.'),
    ('Tier', 'Possible ITES', 'Febrile illness, altered consciousness and an encephalopathic EEG (diffuse slowing or suppression) without syndrome-specific imaging or exclusions.'),
    ('Tier', 'Unclassifiable', 'No documented febrile illness; fever and an encephalopathic EEG without documented altered consciousness; or neither syndrome-specific imaging nor an encephalopathic EEG.'),
    ('Review', 'Flags', 'Clinician label differs from algorithm; exclusion conflicting with ITES-type imaging or clinician ITES label; fever not documented; AESD imaging before day 2; splenial lesion without stated restricted diffusion; clinician label without neuroimaging in the export; qualified alternative diagnosis; cardiac arrest/CPR with ANE/AFCE/ASEM/AESD; altered consciousness not documented for an imaging-defined syndrome or FIRES.'),
], columns=['domain', 'item', 'operational definition'])
CRITERIA

In [ ]:
# Classification summary, agreement with clinician labels, era and outcome profile.
OUT_ITES = OUT
TIER_ORDER = ['Definite ITES', 'ITES-related', 'Possible ITES', 'Sepsis-associated encephalopathy', 'Non-ITES', 'Unclassifiable']
CLASS_ORDER = ['ANE', 'ASEM', 'AFCE', 'AESD', 'MERS', 'HHE', 'FIRES', 'Possible ITES', 'SAE', 'Non-ITES: alternative diagnosis',
               'Non-ITES: encephalitis/meningitis', 'Non-ITES: neonatal encephalopathy', 'Unclassifiable']
ev['era'] = pd.cut(ev.admit, [pd.Timestamp('2000-01-01'), pd.Timestamp('2020-01-01'), pd.Timestamp('2022-12-01'),
                              pd.Timestamp('2023-02-01'), pd.Timestamp('2100-01-01')],
                   labels=['Pre-2020', '2020-Nov 2022', 'Dec 2022-Jan 2023', 'Feb 2023 onward'], right=False)
ev['critical_ama'] = ev.vis.map(idx.set_index('vis').critical_ama)
summary = (ev.groupby('ites_class')
             .agg(tier=('ites_tier', 'first'), N=('vis', 'size'), age_median=('age_y', 'median'), icu=('icu', 'sum'),
                  deaths=('death', 'sum'), death_or_critical_ama=('poor', 'sum'), needs_review=('needs_review', 'sum'))
             .reindex([c for c in CLASS_ORDER if c in set(ev.ites_class)]))
summary['% of cohort'] = [float(rnd(100 * n / len(ev), 1)) for n in summary.N]
summary['age_median'] = [float(rnd(v, 1)) for v in summary.age_median]
agreement = pd.crosstab(ev.ites_class, ev.clinician_label, margins=True).reindex([c for c in CLASS_ORDER if c in set(ev.ites_class)] + ['All'])
by_era = pd.crosstab(ev.ites_class, ev.era).reindex([c for c in CLASS_ORDER if c in set(ev.ites_class)])
excl_counts = pd.DataFrame({'definite (unqualified) diagnosis': {k: int(ev['excl: ' + k].sum()) for k in EXCLUSIONS},
                            'qualified (?) diagnosis only': {k: int(ev['possible excl: ' + k].sum()) for k in EXCLUSIONS}})
lab_map = {'ANE': 'ANE', 'MERS': 'MERS', 'AESD': 'AESD', 'FIRES': 'FIRES', 'SAE': 'SAE'}
spec = ev[ev.clinician_label.isin(lab_map)]
print('Clinician-labeled specific syndromes:', len(spec), '| algorithm agrees:', int((spec.clinician_label == spec.ites_class).sum()))
print('Admissions flagged for review:', int(ev.needs_review.sum()), 'of', len(ev))
summary

In [ ]:
# Exports: classification file (no free text), adjudication worksheet, summary tables.
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, PatternFill, Alignment

FLAG_COLS = ['fever', 'tmax_72h', 'encephalopathy_text', 'seizure_text', 'biphasic_text', 'mri_n', 'mri_first_day', 'ct_n',
             'mri_thal_bilat', 'ct_thal_bilat', 'mri_imp_ane', 'mri_splenial', 'mri_any_dwi', 'mri_subcortical_dwi', 'mri_lobar_wm_dwi',
             'mri_diffuse_edema', 'ct_diffuse_edema', 'mri_hemispheric', 'eeg_n', 'eeg_slow', 'eeg_suppressed', 'eeg_seizure',
             'csf_wbc_max', 'csf_protein_max', 'csf_virus_positive', 'shock', 'septic_shock', 'mof_count'] + [f'mof_{k}' for k in MOF]
cls_out = ev[['patient', 'vis', 'admit', 'age_y', 'clinician_label', 'ites_algorithmic', 'ites_class', 'ites_tier', 'ites_reason',
              'needs_review', 'review_reasons'] + FLAG_COLS].copy()
cls_out['admit'] = cls_out.admit.dt.date
cls_out.to_csv(OUT_ITES / 'ites_classification.csv', index=False, encoding='utf-8-sig')

ENC_DX = re.compile(r'脑病|脑炎|脑膜炎|FIRES|胼胝体|惊厥|癫痫|休克|脓毒|代谢|缺氧|噬血|川崎')
def enc_dx(t):
    items = [x for x in re.split(r'\s*\|\s*', str(t)) if ENC_DX.search(x)]
    return '; '.join(dict.fromkeys(items))[:300]

def evid(row, cols):
    return ' || '.join(str(row[c]) for c in cols if isinstance(row.get(c), str))[:400]

ws = pd.DataFrame({
    'patient': ev.patient, 'vis': ev.vis, 'admission date': ev.admit.dt.date, 'age (years)': ev.age_y.round(2),
    'relevant discharge diagnoses': ev.dx_text.apply(enc_dx), 'clinician label (parsed)': ev.clinician_label,
    'algorithmic class': ev.ites_algorithmic, 'reason': ev.ites_reason, 'review reasons': ev.review_reasons,
    'fever documented': ev.fever.map({True: 'yes', False: 'no'}), 'Tmax first 72 h (C)': ev.tmax_72h,
    'altered consciousness documented': ev.encephalopathy_text.map({True: 'yes', False: 'no'}),
    'seizures': ev.seizure_text.map({True: 'yes', False: 'no'}), 'second seizure cluster cue': ev.biphasic_text.map({True: 'yes', False: 'no'}),
    'MRI first day after admission': ev.mri_first_day.round(1),
    'MRI evidence': ev.apply(lambda r: evid(r, ['mri_thal_bilat_evidence', 'mri_thal_any_evidence', 'mri_imp_ane_evidence', 'mri_diffuse_edema_evidence',
                                                'mri_subcortical_dwi_evidence', 'mri_lobar_wm_dwi_evidence', 'mri_splenial_evidence', 'mri_hemispheric_evidence']), axis=1),
    'MRI normal report': ev.mri_normal.map({True: 'yes', False: 'no'}),
    'CT evidence': ev.apply(lambda r: evid(r, ['ct_thal_bilat_evidence', 'ct_thal_any_evidence', 'ct_diffuse_edema_evidence', 'ct_hemispheric_evidence']), axis=1),
    'EEG diffuse slowing': ev.eeg_slow.map({True: 'yes', False: 'no'}), 'EEG suppression': ev.eeg_suppressed.map({True: 'yes', False: 'no'}),
    'EEG seizure pattern': ev.eeg_seizure.map({True: 'yes', False: 'no'}),
    'CSF WBC max (x10^6/L)': ev.csf_wbc_max, 'CSF protein max (g/L)': ev.csf_protein_max.round(2),
    'CSF viral PCR positive': ev.csf_virus_positive.map({True: 'yes', False: 'no'}),
    'shock (vasoactive within 72 h or diagnosis)': ev.shock.map({True: 'yes', False: 'no'}), 'septic shock diagnosis': ev.septic_shock.map({True: 'yes', False: 'no'}),
    'organ items (of 6)': ev.mof_count, 'organ items present': ev[[f'mof_{k}' for k in MOF]].apply(lambda r: ', '.join(k.replace('mof_', '') for k, v in r.items() if v), axis=1),
    'exclusion diagnoses (unqualified)': ev[[c for c in ev.columns if c.startswith('excl: ')]].apply(lambda r: '; '.join(k.replace('excl: ', '') for k, v in r.items() if v), axis=1),
    'exclusion diagnoses (qualified ?)': ev[[c for c in ev.columns if c.startswith('possible excl: ')]].apply(lambda r: '; '.join(k.replace('possible excl: ', '') for k, v in r.items() if v), axis=1),
    'CPR/arrest documented': ev.cpr.map({True: 'yes', False: 'no'}), 'ICU stay': ev.icu.map({True: 'yes', False: 'no'}),
    'in-hospital death': ev.death.map({True: 'yes', False: 'no'}),
    'adjudicated_class': '', 'adjudicator': '', 'comment': '',
})
ws = ws.assign(_r=~ev.needs_review, _c=ev.ites_class.map({c: i for i, c in enumerate(CLASS_ORDER)})).sort_values(['_r', '_c', 'admission date']).drop(columns=['_r', '_c'])
instructions = pd.DataFrame({'Instructions': [
    'One row per index admission. Rows flagged in "review reasons" are listed first.',
    'Read the evidence columns (and the full record when needed) and enter the final category in "adjudicated_class".',
    'Allowed categories are listed on the sheet "Categories" (drop-down provided). Leave blank to accept the algorithmic class.',
    'Save columns "vis" and "adjudicated_class" as output_ites/ites_adjudicated.csv (UTF-8) and re-run the notebooks.',
    'Definitions follow the 2024 international consensus on infection-triggered encephalopathy syndromes (Dev Med Child Neurol 2024; doi:10.1111/dmcn.16067).',
    'Evidence phrases are short excerpts of radiology/EEG reports and discharge diagnoses; no names are included.']})
categories = pd.DataFrame({'category': CLASS_ORDER, 'tier': [TIER.get(c, 'Non-ITES') for c in CLASS_ORDER]})
WS_PATH = OUT_ITES / 'ITES_adjudication_worksheet.xlsx'
with pd.ExcelWriter(WS_PATH, engine='openpyxl') as xw:
    instructions.to_excel(xw, sheet_name='Instructions', index=False)
    ws.to_excel(xw, sheet_name='Worksheet', index=False)
    categories.to_excel(xw, sheet_name='Categories', index=False)
    CRITERIA.to_excel(xw, sheet_name='Operational_criteria', index=False)
    sh = xw.sheets['Worksheet']
    col = {c: i + 1 for i, c in enumerate(ws.columns)}
    letter = sh.cell(row=1, column=col['adjudicated_class']).column_letter
    dv = DataValidation(type='list', formula1=f"=Categories!$A$2:$A${len(categories) + 1}", allow_blank=True)
    sh.add_data_validation(dv); dv.add(f'{letter}2:{letter}{len(ws) + 1}')
    sh.freeze_panes = 'C2'
    for cell in sh[1]:
        cell.font = Font(bold=True); cell.alignment = Alignment(wrap_text=True, vertical='top')
    fill = PatternFill('solid', fgColor='FFF2CC')
    for name in ['adjudicated_class', 'adjudicator', 'comment']:
        for r in range(1, len(ws) + 2):
            sh.cell(row=r, column=col[name]).fill = fill
    widths = {'relevant discharge diagnoses': 45, 'reason': 40, 'review reasons': 40, 'MRI evidence': 70, 'CT evidence': 50, 'comment': 30}
    for name, i in col.items():
        sh.column_dimensions[sh.cell(row=1, column=i).column_letter].width = widths.get(name, 14)
    xw.sheets['Operational_criteria'].column_dimensions['C'].width = 120
with pd.ExcelWriter(OUT_ITES / 'Tables_ITES.xlsx', engine='openpyxl') as xw:
    summary.reset_index().to_excel(xw, sheet_name='Class_summary', index=False)
    agreement.to_excel(xw, sheet_name='Agreement_vs_clinician')
    by_era.to_excel(xw, sheet_name='Class_by_era')
    excl_counts.to_excel(xw, sheet_name='Exclusion_diagnoses')
    CRITERIA.to_excel(xw, sheet_name='Operational_criteria', index=False)
print('Saved:', [p.name for p in sorted(OUT_ITES.glob('*')) if p.is_file()])